In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD LSTM MODEL ----------------
def build_lstm(input_shape):
    model = Sequential([
        LSTM(96, return_sequences=True, input_shape=input_shape),
        Dropout(0.25),

        LSTM(64, return_sequences=False),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN FUNCTION ----------------
def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch = 16 if len(X_train)>50 else 4

    model = build_lstm((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low, pred_low, f"LSTM {name} Low")

    return (m1, m2)


In [8]:
df = pd.read_csv("crude_oil_2007_to_present.csv")
df = df.rename(columns={df.columns[0]: "Date"})
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",","",regex=False), errors="coerce")

df = df.sort_values("Date").fillna(method="ffill").fillna(method="bfill")
df = df[["Date","Close"]].dropna().reset_index(drop=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_35752\2959239312.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_35752\2959239312.py:9: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.sort_values("Date").fillna(method="ffill").fillna(method="bfill")


In [9]:
df1 = df[df["Date"].dt.year <= 2011]
df2 = df[(df["Date"].dt.year > 2011) & (df["Date"].dt.year <= 2019)]
df3 = df[(df["Date"].dt.year >= 2020) & (df["Date"].dt.year <= 2022)]
df4 = df[df["Date"].dt.year > 2022]

splits = [
    ("2007–2011", df1),
    ("2011–2019", df2),
    ("2020–2022", df3),
    ("2022–Latest", df4)
]

In [10]:
import pandas as pd

df = pd.read_csv("crude_oil_2007_to_present.csv")
print(df.columns)


Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')


In [18]:
df.head()

,Price,Close,High,Low,Open,Volume
0,2007-01-03,58.31999969482422,60.970001220703125,57.720001220703125,60.95000076293945,248547
1,2007-01-04,55.59000015258789,58.58000183105469,55.38999938964844,58.349998474121094,324010
2,2007-01-05,56.310001373291016,56.400001525878906,54.900001525878906,55.40999984741211,250130
3,2007-01-08,56.09000015258789,57.720001220703125,55.099998474121094,56.2400016784668,298727
4,2007-01-09,55.63999938964844,56.20000076293945,53.880001068115234,56.119998931884766,333330


In [2]:
import pandas as pd
import numpy as np

# ---------------- LOAD DATA ----------------
df = pd.read_csv("forex_usd_inr_2007_to_present.csv")

# The first column is actually the Date column, rename it properly
df = df.rename(columns={df.columns[0]: "Date"})

# Convert Date column to proper datetime
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Drop invalid rows if any occurred during shifting
df = df.dropna(subset=["Date"])

# Convert numeric columns safely
for col in df.columns:
    if col != "Date":
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing values created during conversion
df = df.ffill().bfill()

# Set date as index
df = df.set_index("Date")

print(df.head())
print(df.dtypes)

# ---------------- CREATE REGIME-BASED SPLITS ----------------
df1 = df.loc[: "2010-12-31"]              # Pre-2011
df2 = df.loc["2011-01-01":"2019-12-31"]   # 2011–2019
df3 = df.loc["2020-01-01":"2021-12-31"]   # COVID period
df4 = df.loc["2022-01-01":]               # 2022 onward

# Print sizes
print("\nSizes of each segment:")
print("df1 (2007–2010):", len(df1))
print("df2 (2011–2019):", len(df2))
print("df3 (2020–2021):", len(df3))
print("df4 (2022–present):", len(df4))


                Close       High        Low       Open  Volume
Date                                                          
2007-01-01  44.215000  44.223999  44.040001  44.220001       0
2007-01-02  44.122002  44.220001  44.075001  44.212002       0
2007-01-03  44.111000  44.410000  44.085999  44.119999       0
2007-01-04  44.095001  44.480000  44.095001  44.119999       0
2007-01-05  44.139999  44.330002  44.095001  44.095001       0
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Sizes of each segment:
df1 (2007–2010): 1021
df2 (2011–2019): 2342
df3 (2020–2021): 523
df4 (2022–present): 1003


C:\Users\ASUS\AppData\Local\Temp\ipykernel_28784\2624355148.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Dictionary to store all results
results = {
    "XGBoost": {},
    "CatBoost": {},
    "TCN": {},
    "ARIMA": {},
    "SARIMA": {},
    "BiLSTM": {},
    "ConvLSTM": {}
}

datasets = {
    "df1": df1,
    "df2": df2,
    "df3": df3,
    "df4": df4
}

def prepare_data(df):
    df = df.copy()

 

    # Multi-output targets
    y = df[["High", "Low"]]           # predicting both
    X = df.drop(["High", "Low"], axis=1)

    return train_test_split(X, y, test_size=0.2, random_state=42)


In [5]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

def evaluate_model(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # Avoid divide-by-zero in MAPE
    non_zero_idx = y_true != 0
    mape = np.mean(np.abs((y_true[non_zero_idx] - y_pred[non_zero_idx]) / y_true[non_zero_idx])) * 100

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)

    return {
        "MAPE": mape,
        "RMSE": rmse,
        "R2": r2,
        "MAE": mae
    }


In [6]:
from xgboost import XGBRegressor

for name, df in datasets.items():
    X_train, X_test, y_train, y_test = prepare_data(df)

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8
    )
    
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    results["XGBoost"][name] = {
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "R2": r2_score(y_test, preds)
    }

results["XGBoost"]


{'df1': {'RMSE': 0.17127358813619165, 'R2': 0.997622013092041},
 'df2': {'RMSE': 0.3508423741361993, 'R2': 0.9978656768798828},
 'df3': {'RMSE': 0.24607419889988785, 'R2': 0.974882185459137},
 'df4': {'RMSE': 0.2577325379027786, 'R2': 0.9934461116790771}}

In [7]:
from xgboost import XGBRegressor

for name, df in datasets.items():
    X_train, X_test, y_train, y_test = prepare_data(df)

    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results["XGBoost"][name] = evaluate_model(y_test, preds)
    


In [6]:
results["XGBoost"]

{'df1': {'MAPE': 0.22769651151016376,
  'RMSE': 0.17038744257163685,
  'R2': 0.997644944726608,
  'MAE': 0.1031679572128669},
 'df2': {'MAPE': 0.2820860237767482,
  'RMSE': 0.35418035579949486,
  'R2': 0.9978249081453627,
  'MAE': 0.1675409955510707},
 'df3': {'MAPE': 0.23356172316595916,
  'RMSE': 0.2526898219680648,
  'R2': 0.9734257269249635,
  'MAE': 0.1745985848563058},
 'df4': {'MAPE': 0.16850131220048392,
  'RMSE': 0.25912511150603484,
  'R2': 0.9933747502456891,
  'MAE': 0.13976724349444186}}

In [8]:
from statsmodels.tsa.arima.model import ARIMA

for name, df in datasets.items():
    y = df["High"]  # choose High or Low separately

    train_size = int(len(y) * 0.8)
    train, test = y[:train_size], y[train_size:]

    model = ARIMA(train, order=(2,1,2))
    model_fit = model.fit()
    preds = model_fit.forecast(len(test))

    results["ARIMA"][name] = evaluate_model(test, preds)


d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
d:\Anacondaa\envs\tfenv\lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
d:\Anacondaa\envs\tf

In [9]:
results["ARIMA"]

{'df1': {'MAPE': 10.545246377799621,
  'RMSE': 163.67499796650466,
  'R2': -2.3108132735427467,
  'MAE': 138.48302705491383},
 'df2': {'MAPE': 6.273673198732937,
  'RMSE': 105.37688196699078,
  'R2': -0.010481944895258888,
  'MAE': 85.51033689102611},
 'df3': {'MAPE': 2.7951623016531237,
  'RMSE': 57.830059938411516,
  'R2': -3.0894167596476496,
  'MAE': 50.64446880873578},
 'df4': {'MAPE': 15.45336017495487,
  'RMSE': 658.6615141525367,
  'R2': -2.48595968890244,
  'MAE': 556.2142402588826}}

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD BiLSTM MODEL ----------------
def build_bilstm(input_shape):
    model = Sequential([

        Bidirectional(LSTM(96, return_sequences=True), input_shape=input_shape),
        Dropout(0.25),

        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN FUNCTION ----------------
def run_BiLSTM(df, name):

    print(f"\n================ BiLSTM for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch = 16 if len(X_train)>50 else 4

    model = build_bilstm((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"BiLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"BiLSTM {name} Low")

    return (m1, m2)


In [8]:
# ================= RUN BILSTM ON ALL DATASETS ================= #

# df1, df2, df3, df4 must already be loaded before running this cell.

print("\n\n====== RUNNING BiLSTM ON ALL DATASETS ======\n")

results_bilstm = {}

for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training BiLSTM for {name} -----")
    try:
        res = run_BiLSTM(df, name)
        results_bilstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL BiLSTM RESULTS =============\n")
results_bilstm




====== RUNNING BiLSTM ON ALL DATASETS ======



----- Training BiLSTM for DF1 -----

================ BiLSTM for DF1 ================
Epoch 1/60


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - loss: 0.1302 - val_loss: 0.0174 - learning_rate: 0.0010
Epoch 2/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0364 - val_loss: 0.0206 - learning_rate: 0.0010
Epoch 3/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0284 - val_loss: 0.0128 - learning_rate: 0.0010
Epoch 4/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0283 - val_loss: 0.0090 - learning_rate: 0.0010
Epoch 5/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0244 - val_loss: 0.0114 - learning_rate: 0.0010
Epoch 6/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0199 - val_loss: 0.0081 - learning_rate: 0.0010
Epoch 7/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0205 - val_loss: 0.0098 - learning_rate: 0.0010
Epoch 8/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0171 - val_loss: 0.0091 - learning_rate: 0.0010
Epoch 9/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0165 - val_loss: 0.0074 - learning_rate: 0.0010
Epoch 10/60
53/53 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


123/123 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0511 - val_loss: 0.0205 - learning_rate: 0.0010
Epoch 2/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0194 - val_loss: 0.0237 - learning_rate: 0.0010
Epoch 3/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0160 - val_loss: 0.0074 - learning_rate: 0.0010
Epoch 4/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0135 - val_loss: 0.0074 - learning_rate: 0.0010
Epoch 5/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0121 - val_loss: 0.0043 - learning_rate: 0.0010
Epoch 6/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.0117 - val_loss: 0.0049 - learning_rate: 0.0010
Epoch 7/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0105 - val_loss: 0.0083 - learning_rate: 0.0010
Epoch 8/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0100 - val_loss: 0.0086 - learning_rate: 0.0010
Epoch 9/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.0092 - val_loss: 0.0130 - learning_rate: 5.0000e-04
Epoch 10

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


27/27 ━━━━━━━━━━━━━━━━━━━━ 4s 29ms/step - loss: 0.3355 - val_loss: 0.1564 - learning_rate: 0.0010
Epoch 2/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.1044 - val_loss: 0.0906 - learning_rate: 0.0010
Epoch 3/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0910 - val_loss: 0.0880 - learning_rate: 0.0010
Epoch 4/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0835 - val_loss: 0.0725 - learning_rate: 0.0010
Epoch 5/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0769 - val_loss: 0.0699 - learning_rate: 0.0010
Epoch 6/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0714 - val_loss: 0.0840 - learning_rate: 0.0010
Epoch 7/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0720 - val_loss: 0.0615 - learning_rate: 0.0010
Epoch 8/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0746 - val_loss: 0.0787 - learning_rate: 0.0010
Epoch 9/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0653 - val_loss: 0.0697 - learning_rate: 0.0010
Epoch 10/60
27/27 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - loss: 0.1150 - val_loss: 0.1819 - learning_rate: 0.0010
Epoch 2/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0390 - val_loss: 0.1097 - learning_rate: 0.0010
Epoch 3/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0317 - val_loss: 0.1919 - learning_rate: 0.0010
Epoch 4/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0356 - val_loss: 0.1798 - learning_rate: 0.0010
Epoch 5/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0329 - val_loss: 0.1154 - learning_rate: 0.0010
Epoch 6/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.0276 - val_loss: 0.1063 - learning_rate: 5.0000e-04
Epoch 7/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0237 - val_loss: 0.1466 - learning_rate: 5.0000e-04
Epoch 8/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0230 - val_loss: 0.1080 - learning_rate: 5.0000e-04
Epoch 9/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0245 - val_loss: 0.1134 - learning_rate: 5.0000e-04
Epoch 10/60
52

{'DF1': {'High': {'MAE': 0.19755759796538971,
   'RMSE': 0.2530512257736174,
   'R2': 0.9304718440176201,
   'MAPE': 0.43145083670438883},
  'Low': {'MAE': 0.17774386220164098,
   'RMSE': 0.2652892375188861,
   'R2': 0.9252201085588544,
   'MAPE': 0.3929495621000152}},
 'DF2': {'High': {'MAE': 0.3556367484006015,
   'RMSE': 0.48584199976846365,
   'R2': 0.8655259200457812,
   'MAPE': 0.49785983983857524},
  'Low': {'MAE': 0.33955125375227496,
   'RMSE': 0.4455203478321635,
   'R2': 0.8774443158825672,
   'MAPE': 0.4812499004601487}},
 'DF3': {'High': {'MAE': 0.22982730141168908,
   'RMSE': 0.2816546716185565,
   'R2': 0.8256253450018477,
   'MAPE': 0.3058130882075704},
  'Low': {'MAE': 0.209571162356606,
   'RMSE': 0.2602527266537547,
   'R2': 0.8333167546316583,
   'MAPE': 0.2802017368442496}},
 'DF4': {'High': {'MAE': 0.48046698159729406,
   'RMSE': 0.5657221723845931,
   'R2': 0.8090084327186239,
   'MAPE': 0.5528503077294359},
  'Low': {'MAE': 0.6033492940940605,
   'RMSE': 1.20137

In [9]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tcn import TCN


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred)/y_true))*100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae,3))
    print("RMSE:",  round(rmse,3))
    print("R2  :",  round(r2,3))
    print("MAPE:", round(mape,2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD TCN MODEL ----------------
def build_tcn(input_shape):
    model = Sequential([

        TCN(nb_filters=64,
            kernel_size=3,
            dilations=[1, 2, 4, 8, 16],
            dropout_rate=0.25,
            return_sequences=False,
            activation='relu',
            input_shape=input_shape),

        Dense(32, activation='relu'),
        Dropout(0.25),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model


# ---------------- MAIN TCN FUNCTION ----------------
def run_TCN(df, name):

    print(f"\n================ TCN for {name} ================")

    # Features and target
    X = df[['Open','High','Low','Close']].values
    y = df[['High','Low']].values

    seq_len = 25
    train_size = max(int(len(df)*0.85), seq_len+3)
    train_size = min(train_size, len(df)-2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Sequence creation
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len

    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    batch = 16 if len(X_train)>50 else 4

    model = build_tcn((seq_len, X_train.shape[2]))

    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch,
        validation_data=(X_test, y_test) if len(X_test)>1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:,0], pred[:,1]
    true_high, true_low = y_true[:,0], y_true[:,1]

    m1 = metrics(true_high, pred_high, f"TCN {name} High")
    m2 = metrics(true_low,  pred_low,  f"TCN {name} Low")

    return (m1, m2)


In [10]:
results_tcn = {}

for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training TCN for {name} -----")
    try:
        res = run_TCN(df, name)
        results_tcn[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

results_tcn




----- Training TCN for DF1 -----

================ TCN for DF1 ================


d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)



Epoch 1/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 24.4102 - val_loss: 0.1103 - learning_rate: 0.0010
Epoch 2/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.4618 - val_loss: 0.0657 - learning_rate: 0.0010
Epoch 3/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.0883 - val_loss: 0.0631 - learning_rate: 0.0010
Epoch 4/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.8839 - val_loss: 0.0483 - learning_rate: 0.0010
Epoch 5/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.7322 - val_loss: 0.0375 - learning_rate: 0.0010
Epoch 6/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6406 - val_loss: 0.0340 - learning_rate: 0.0010
Epoch 7/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6191 - val_loss: 0.0334 - learning_rate: 0.0010
Epoch 8/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.5351 - val_loss: 0.0381 - learning_rate: 0.0010
Epoch 9/60
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.4736 - val_loss: 0.0318 - learning_rate: 0.0010
Epoch 10/60
53/53 ━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


123/123 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 12.3860 - val_loss: 1.4690 - learning_rate: 0.0010
Epoch 2/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.7376 - val_loss: 1.0554 - learning_rate: 0.0010
Epoch 3/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.5048 - val_loss: 0.5575 - learning_rate: 0.0010
Epoch 4/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3962 - val_loss: 0.3311 - learning_rate: 0.0010
Epoch 5/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3172 - val_loss: 0.3044 - learning_rate: 0.0010
Epoch 6/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2737 - val_loss: 0.4374 - learning_rate: 0.0010
Epoch 7/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2424 - val_loss: 0.2406 - learning_rate: 0.0010
Epoch 8/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2629 - val_loss: 0.1738 - learning_rate: 0.0010
Epoch 9/60
123/123 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.2023 - val_loss: 0.2597 - learning_rate: 0.0010
Epoch 10/60
123/123 

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


27/27 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 20.6086 - val_loss: 0.1814 - learning_rate: 0.0010
Epoch 2/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 2.4909 - val_loss: 0.2490 - learning_rate: 0.0010
Epoch 3/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.3524 - val_loss: 0.2193 - learning_rate: 0.0010
Epoch 4/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.0190 - val_loss: 0.1987 - learning_rate: 0.0010
Epoch 5/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.9923 - val_loss: 0.2399 - learning_rate: 5.0000e-04
Epoch 6/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.8617 - val_loss: 0.2547 - learning_rate: 5.0000e-04
Epoch 7/60
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.7436 - val_loss: 0.2613 - learning_rate: 5.0000e-04
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step

===== TCN DF3 High =====
MAE : 0.481
RMSE: 0.575
R2  : 0.274
MAPE: 0.64 %

===== TCN DF3 Low =====
MAE : 0.438
RMSE: 0.566
R2  : 0.212
MAPE: 0.59 %


----- Training TCN for DF4 -----

============

d:\Anacondaa\envs\tfenv\lib\site-packages\tcn\tcn.py:268: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super(TCN, self).__init__(**kwargs)


52/52 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 29.6244 - val_loss: 0.1292 - learning_rate: 0.0010
Epoch 2/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.5909 - val_loss: 0.1141 - learning_rate: 0.0010
Epoch 3/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.2954 - val_loss: 0.1707 - learning_rate: 0.0010
Epoch 4/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.9978 - val_loss: 0.3162 - learning_rate: 0.0010
Epoch 5/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6806 - val_loss: 0.3761 - learning_rate: 0.0010
Epoch 6/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6306 - val_loss: 0.4811 - learning_rate: 5.0000e-04
Epoch 7/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6205 - val_loss: 0.4414 - learning_rate: 5.0000e-04
Epoch 8/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6406 - val_loss: 0.4724 - learning_rate: 5.0000e-04
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step

===== TCN DF4 High =====
MAE : 0.46
RMSE: 0.586
R2  : 0.795
MAPE: 0.53 %

===== TCN DF4 

{'DF1': {'High': {'MAE': 0.2963664562671216,
   'RMSE': 0.36705173862582174,
   'R2': 0.8537154730602954,
   'MAPE': 0.6479131030944418},
  'Low': {'MAE': 0.32535253252301866,
   'RMSE': 0.43424630130878294,
   'R2': 0.799636862647999,
   'MAPE': 0.7121003907400064}},
 'DF2': {'High': {'MAE': 1.9548481594432483,
   'RMSE': 2.0357751030259474,
   'R2': -1.3610656760231583,
   'MAPE': 2.7389145637329246},
  'Low': {'MAE': 1.8978082700209185,
   'RMSE': 1.9845185696184857,
   'R2': -1.431692777130381,
   'MAPE': 2.6790730082732512}},
 'DF3': {'High': {'MAE': 0.48087648802165744,
   'RMSE': 0.5747959471670478,
   'R2': 0.27376556415370856,
   'MAPE': 0.6392723398250584},
  'Low': {'MAE': 0.4382505779024921,
   'RMSE': 0.5659967263202604,
   'R2': 0.21163159737709936,
   'MAPE': 0.5883313853788824}},
 'DF4': {'High': {'MAE': 0.46030648338873653,
   'RMSE': 0.5856902217612063,
   'R2': 0.7952877938508347,
   'MAPE': 0.5301343014355172},
  'Low': {'MAE': 0.649761679946192,
   'RMSE': 1.280073

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD LSTM MODEL ----------------
def build_lstm(input_shape):

    model = Sequential([

        LSTM(96, return_sequences=True, input_shape=input_shape),
        Dropout(0.25),

        LSTM(64, return_sequences=False),
        Dropout(0.25),

        Dense(32, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_LSTM(df, name):

    print(f"\n================ LSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # Adaptive batch size
    batch_size = 16 if len(X_train) > 50 else 4

    # Build model
    model = build_lstm((seq_len, X_train.shape[2]))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train
    model.fit(
        X_train, y_train,
        epochs=60,
        batch_size=batch_size,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Prediction
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"LSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"LSTM {name} Low")

    return (m1, m2)



# ================= RUN LSTM ON ALL DATASETS ================= #

print("\n\n====== RUNNING LSTM ON ALL DATASETS ======\n")

results_lstm = {}

# df1, df2, df3, df4 must already be defined
for df, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training LSTM for {name} -----")
    try:
        res = run_LSTM(df, name)
        results_lstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL LSTM RESULTS =============\n")
results_lstm




====== RUNNING LSTM ON ALL DATASETS ======



----- Training LSTM for DF1 -----

================ LSTM for DF1 ================
Epoch 1/60


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


52/52 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 0.1669 - val_loss: 0.4689 - learning_rate: 0.0010
Epoch 2/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0449 - val_loss: 0.4615 - learning_rate: 0.0010
Epoch 3/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0406 - val_loss: 0.4098 - learning_rate: 0.0010
Epoch 4/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0331 - val_loss: 0.2023 - learning_rate: 0.0010
Epoch 5/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0346 - val_loss: 0.3300 - learning_rate: 0.0010
Epoch 6/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0317 - val_loss: 0.4005 - learning_rate: 0.0010
Epoch 7/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0308 - val_loss: 0.1467 - learning_rate: 0.0010
Epoch 8/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0257 - val_loss: 0.1202 - learning_rate: 0.0010
Epoch 9/60
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0260 - val_loss: 0.1954 - learning_rate: 0.0010
Epoch 10/60
52/52 ━━━━━━━━━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


119/119 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.1022 - val_loss: 0.0093 - learning_rate: 0.0010
Epoch 2/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0326 - val_loss: 0.0082 - learning_rate: 0.0010
Epoch 3/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0299 - val_loss: 0.0067 - learning_rate: 0.0010
Epoch 4/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0251 - val_loss: 0.0058 - learning_rate: 0.0010
Epoch 5/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0243 - val_loss: 0.0062 - learning_rate: 0.0010
Epoch 6/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0205 - val_loss: 0.0054 - learning_rate: 0.0010
Epoch 7/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0193 - val_loss: 0.0059 - learning_rate: 0.0010
Epoch 8/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0202 - val_loss: 0.0061 - learning_rate: 0.0010
Epoch 9/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0185 - val_loss: 0.0046 - learning_rate: 0.0010
Epoch 10/60
119/119 ━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


26/26 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.3256 - val_loss: 0.0607 - learning_rate: 0.0010
Epoch 2/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1365 - val_loss: 0.0467 - learning_rate: 0.0010
Epoch 3/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1277 - val_loss: 0.0414 - learning_rate: 0.0010
Epoch 4/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1110 - val_loss: 0.0312 - learning_rate: 0.0010
Epoch 5/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1012 - val_loss: 0.0338 - learning_rate: 0.0010
Epoch 6/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0972 - val_loss: 0.0342 - learning_rate: 0.0010
Epoch 7/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0913 - val_loss: 0.0258 - learning_rate: 0.0010
Epoch 8/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0867 - val_loss: 0.0265 - learning_rate: 0.0010
Epoch 9/60
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0868 - val_loss: 0.0257 - learning_rate: 0.0010
Epoch 10/60
26/26 ━━━━━━━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.1660 - val_loss: 2.0764 - learning_rate: 0.0010
Epoch 2/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0322 - val_loss: 1.3791 - learning_rate: 0.0010
Epoch 3/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0321 - val_loss: 1.1085 - learning_rate: 0.0010
Epoch 4/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0251 - val_loss: 1.0626 - learning_rate: 0.0010
Epoch 5/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0281 - val_loss: 1.1118 - learning_rate: 0.0010
Epoch 6/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0228 - val_loss: 0.9118 - learning_rate: 0.0010
Epoch 7/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0250 - val_loss: 0.9922 - learning_rate: 0.0010
Epoch 8/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0186 - val_loss: 0.6893 - learning_rate: 0.0010
Epoch 9/60
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0175 - val_loss: 0.5758 - learning_rate: 0.0010
Epoch 10/60
50/50 ━━━━━━━━━━━━━━━━━━━━

{'DF1': {'High': {'MAE': 27.711849011872943,
   'RMSE': 36.612216197810056,
   'R2': 0.7779693044472344,
   'MAPE': 2.0616771150482345},
  'Low': {'MAE': 30.440773411800986,
   'RMSE': 40.23127150610587,
   'R2': 0.7099671594284663,
   'MAPE': 2.2806980065377926}},
 'DF2': {'High': {'MAE': 7.8130722943474264,
   'RMSE': 10.150959378588546,
   'R2': 0.991843878562979,
   'MAPE': 0.5755689500424426},
  'Low': {'MAE': 7.669440056295956,
   'RMSE': 9.802478042318493,
   'R2': 0.9919198298129385,
   'MAPE': 0.5685337544567696}},
 'DF3': {'High': {'MAE': 11.049513967413658,
   'RMSE': 14.407605987057911,
   'R2': 0.7735470372475217,
   'MAPE': 0.6143578783479722},
  'Low': {'MAE': 11.226690995065793,
   'RMSE': 14.906082387087272,
   'R2': 0.7894193460372172,
   'MAPE': 0.6308971878889665}},
 'DF4': {'High': {'MAE': 159.1779651380565,
   'RMSE': 244.28554198659765,
   'R2': 0.32621047446654816,
   'MAPE': 4.137783062405376},
  'Low': {'MAE': 167.80884959599743,
   'RMSE': 262.3248743348272,


In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)



# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape



# ---------------- BUILD ConvLSTM MODEL ----------------
def build_convlstm(input_shape):

    model = Sequential([

        ConvLSTM2D(
            filters=64,
            kernel_size=(1, 2),
            return_sequences=True,
            activation='tanh',
            padding='same',
            input_shape=input_shape
        ),
        Dropout(0.25),

        ConvLSTM2D(
            filters=32,
            kernel_size=(1, 2),
            return_sequences=False,
            activation='tanh',
            padding='same'
        ),
        Dropout(0.25),

        Flatten(),

        Dense(64, activation='relu'),
        Dense(2)
    ])

    model.compile(optimizer='adam', loss='mse')
    return model



# ---------------- MAIN FUNCTION ----------------
def run_ConvLSTM(df, name):

    print(f"\n================ ConvLSTM for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequence creation
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split into train/test
    split = train_size - seq_len
    X_train_seq, X_test_seq = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # ConvLSTM requires 5D input: (batch, seq_len, spatial_x, spatial_y, channels)
    X_train = X_train_seq.reshape(-1, seq_len, 1, X_train_raw.shape[1], 1)
    X_test  = X_test_seq.reshape(-1, seq_len, 1, X_train_raw.shape[1], 1)

    # Adaptive batch size
    batch_size = 8 if len(X_train) > 50 else 2

    # Build model
    model = build_convlstm((seq_len, 1, X_train_raw.shape[1], 1))

    # Callbacks
    es = EarlyStopping(patience=6, restore_best_weights=True)
    lr = ReduceLROnPlateau(factor=0.5, patience=3)

    # Train model
    model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=batch_size,
        validation_data=(X_test, y_test) if len(X_test) > 1 else None,
        callbacks=[es, lr],
        verbose=1
    )

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"ConvLSTM {name} High")
    m2 = metrics(true_low,  pred_low,  f"ConvLSTM {name} Low")

    return (m1, m2)



# ================= RUN ConvLSTM ON ALL DATASETS ================= #
print("\n\n====== RUNNING ConvLSTM ON ALL DATASETS ======\n")

results_convlstm = {}

for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training ConvLSTM for {name} -----")
    try:
        res = run_ConvLSTM(df_regime, name)
        results_convlstm[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL ConvLSTM RESULTS =============\n")
results_convlstm




====== RUNNING ConvLSTM ON ALL DATASETS ======



----- Training ConvLSTM for DF1 -----

================ ConvLSTM for DF1 ================
Epoch 1/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


106/106 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 0.0890 - val_loss: 0.0205 - learning_rate: 0.0010
Epoch 2/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0305 - val_loss: 0.0114 - learning_rate: 0.0010
Epoch 3/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0244 - val_loss: 0.0229 - learning_rate: 0.0010
Epoch 4/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.0228 - val_loss: 0.0197 - learning_rate: 0.0010
Epoch 5/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0189 - val_loss: 0.0111 - learning_rate: 0.0010
Epoch 6/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.0168 - val_loss: 0.0093 - learning_rate: 0.0010
Epoch 7/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0145 - val_loss: 0.0086 - learning_rate: 0.0010
Epoch 8/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.0162 - val_loss: 0.0071 - learning_rate: 0.0010
Epoch 9/50
106/106 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0160 - val_loss: 0.0091 - learning_rate: 0.0010
Epoch 10/50


d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


246/246 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - loss: 0.0421 - val_loss: 0.0082 - learning_rate: 0.0010
Epoch 2/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 0.0147 - val_loss: 0.0070 - learning_rate: 0.0010
Epoch 3/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - loss: 0.0137 - val_loss: 0.0069 - learning_rate: 0.0010
Epoch 4/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 7s 26ms/step - loss: 0.0111 - val_loss: 0.0317 - learning_rate: 0.0010
Epoch 5/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - loss: 0.0109 - val_loss: 0.0096 - learning_rate: 0.0010
Epoch 6/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - loss: 0.0088 - val_loss: 0.0045 - learning_rate: 5.0000e-04
Epoch 7/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - loss: 0.0080 - val_loss: 0.0062 - learning_rate: 5.0000e-04
Epoch 8/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - loss: 0.0076 - val_loss: 0.0040 - learning_rate: 5.0000e-04
Epoch 9/50
246/246 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - loss: 0.0080 - val_loss: 0.0033 - learning_rate: 5.0000

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


53/53 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - loss: 0.2513 - val_loss: 0.1424 - learning_rate: 0.0010
Epoch 2/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.1251 - val_loss: 0.1167 - learning_rate: 0.0010
Epoch 3/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.1134 - val_loss: 0.0963 - learning_rate: 0.0010
Epoch 4/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0952 - val_loss: 0.0920 - learning_rate: 0.0010
Epoch 5/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0822 - val_loss: 0.0585 - learning_rate: 0.0010
Epoch 6/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0796 - val_loss: 0.0584 - learning_rate: 0.0010
Epoch 7/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - loss: 0.0715 - val_loss: 0.0488 - learning_rate: 0.0010
Epoch 8/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0730 - val_loss: 0.0534 - learning_rate: 0.0010
Epoch 9/50
53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.0684 - val_loss: 0.0448 - learning_rate: 0.0010
Epoch 10/50
53/53 ━━━━━━━━━━━━

d:\Anacondaa\envs\tfenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


104/104 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 0.1059 - val_loss: 0.1115 - learning_rate: 0.0010
Epoch 2/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 0.0342 - val_loss: 0.1260 - learning_rate: 0.0010
Epoch 3/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.0331 - val_loss: 0.0911 - learning_rate: 0.0010
Epoch 4/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0299 - val_loss: 0.0947 - learning_rate: 0.0010
Epoch 5/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - loss: 0.0263 - val_loss: 0.0940 - learning_rate: 0.0010
Epoch 6/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.0256 - val_loss: 0.1127 - learning_rate: 0.0010
Epoch 7/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.0241 - val_loss: 0.0861 - learning_rate: 5.0000e-04
Epoch 8/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - loss: 0.0226 - val_loss: 0.0889 - learning_rate: 5.0000e-04
Epoch 9/50
104/104 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.0222 - val_loss: 0.0834 - learning_rate: 5.0000e-04


{'DF1': {'High': {'MAE': 0.2128720964704247,
   'RMSE': 0.2627061816736894,
   'R2': 0.9250650529319081,
   'MAPE': 0.46450911699236686},
  'Low': {'MAE': 0.1791481414398591,
   'RMSE': 0.2701406775841637,
   'R2': 0.922460046355279,
   'MAPE': 0.3964368302510803}},
 'DF2': {'High': {'MAE': 0.28289023312655365,
   'RMSE': 0.3732852665742635,
   'R2': 0.9206165137270963,
   'MAPE': 0.39701672529920834},
  'Low': {'MAE': 0.2888095812364058,
   'RMSE': 0.3763694044515917,
   'R2': 0.9125364714080387,
   'MAPE': 0.40960479015561974}},
 'DF3': {'High': {'MAE': 0.19306173203866692,
   'RMSE': 0.24089387478975333,
   'R2': 0.872443980552871,
   'MAPE': 0.2567402857112477},
  'Low': {'MAE': 0.17114190210269975,
   'RMSE': 0.21413639339150972,
   'R2': 0.88715499538049,
   'MAPE': 0.22887718436915197}},
 'DF4': {'High': {'MAE': 0.2886331318230029,
   'RMSE': 0.38570231340140704,
   'R2': 0.9112205849518831,
   'MAPE': 0.33236874863434596},
  'Low': {'MAE': 0.42426138366295013,
   'RMSE': 1.0808

In [17]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor


# ---------------- SEQUENCE CREATION ----------------
def create_seq(X, y, seq_len=25):
    X_seq, y_seq = [], []
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i].flatten())  # Flatten for CatBoost
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)


# ---------------- METRICS ----------------
def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true,  y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    print(f"\n===== {label} =====")
    print("MAE :",  round(mae, 3))
    print("RMSE:",  round(rmse, 3))
    print("R2  :",  round(r2, 3))
    print("MAPE:", round(mape, 2), "%")

    return mae, rmse, r2, mape


# ---------------- BUILD & RUN CATBOOST MODEL ----------------
def run_CatBoost(df, name):

    print(f"\n================ CatBoost for {name} ================")

    # Features and target
    X = df[['Open', 'High', 'Low', 'Close']].values
    y = df[['High', 'Low']].values

    seq_len = 25
    train_size = max(int(len(df) * 0.85), seq_len + 3)
    train_size = min(train_size, len(df) - 2)

    X_train_raw = X[:train_size]
    X_test_raw  = X[train_size:]

    y_train_raw = y[:train_size]
    y_test_raw  = y[train_size:]

    # Scaling
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train_raw)
    X_test_scaled  = scaler_X.transform(X_test_raw)

    y_train_scaled = scaler_y.fit_transform(y_train_raw)
    y_test_scaled  = scaler_y.transform(y_test_raw)

    # Merge for sequence creation
    X_scaled = np.vstack([X_train_scaled, X_test_scaled])
    y_scaled = np.vstack([y_train_scaled, y_test_scaled])

    # Create sequences
    X_seq, y_seq = create_seq(X_scaled, y_scaled, seq_len)

    # Split into train/test
    split = train_size - seq_len
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]

    # ---------------- CatBoost Model ----------------
    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiRMSE',
        verbose=False
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    pred_scaled = model.predict(X_test)
    pred = scaler_y.inverse_transform(pred_scaled)
    y_true = scaler_y.inverse_transform(y_test)

    pred_high, pred_low = pred[:, 0], pred[:, 1]
    true_high, true_low = y_true[:, 0], y_true[:, 1]

    m1 = metrics(true_high, pred_high, f"CatBoost {name} High")
    m2 = metrics(true_low,  pred_low,  f"CatBoost {name} Low")

    return (m1, m2)



# ================= RUN CATBOOST ON ALL DATASETS ================= #

print("\n\n====== RUNNING CatBoost ON ALL DATASETS ======\n")

results_catboost = {}

# df1, df2, df3, df4 must already be created before this
for df_regime, name in zip([df1, df2, df3, df4], ["DF1", "DF2", "DF3", "DF4"]):
    print(f"\n\n----- Training CatBoost for {name} -----")
    try:
        res = run_CatBoost(df_regime, name)
        results_catboost[name] = {
            "High": {
                "MAE":  res[0][0],
                "RMSE": res[0][1],
                "R2":   res[0][2],
                "MAPE": res[0][3]
            },
            "Low": {
                "MAE":  res[1][0],
                "RMSE": res[1][1],
                "R2":   res[1][2],
                "MAPE": res[1][3]
            }
        }
    except Exception as e:
        print(f"Error in {name}: {e}")

print("\n\n============= FINAL CatBoost RESULTS =============\n")
results_catboost




====== RUNNING CatBoost ON ALL DATASETS ======



----- Training CatBoost for DF1 -----

================ CatBoost for DF1 ================

===== CatBoost DF1 High =====
MAE : 84.849
RMSE: 111.075
R2  : -1.044
MAPE: 6.26 %

===== CatBoost DF1 Low =====
MAE : 75.497
RMSE: 99.879
R2  : -0.788
MAPE: 5.63 %


----- Training CatBoost for DF2 -----

================ CatBoost for DF2 ================

===== CatBoost DF2 High =====
MAE : 9.059
RMSE: 12.161
R2  : 0.988
MAPE: 0.66 %

===== CatBoost DF2 Low =====
MAE : 8.768
RMSE: 11.644
R2  : 0.989
MAPE: 0.64 %


----- Training CatBoost for DF3 -----

================ CatBoost for DF3 ================

===== CatBoost DF3 High =====
MAE : 11.686
RMSE: 14.956
R2  : 0.756
MAPE: 0.65 %

===== CatBoost DF3 Low =====
MAE : 10.229
RMSE: 14.904
R2  : 0.789
MAPE: 0.57 %


----- Training CatBoost for DF4 -----

================ CatBoost for DF4 ================

===== CatBoost DF4 High =====
MAE : 344.661
RMSE: 455.209
R2  : -1.34
MAPE: 9.13 %

===== C

{'DF1': {'High': {'MAE': 84.84908816200532,
   'RMSE': 111.07452851478263,
   'R2': -1.0435719824771699,
   'MAPE': 6.2580634337170995},
  'Low': {'MAE': 75.49663544262633,
   'RMSE': 99.87863667096451,
   'R2': -0.7875774609997899,
   'MAPE': 5.629347419939372}},
 'DF2': {'High': {'MAE': 9.059494688965536,
   'RMSE': 12.160641493685477,
   'R2': 0.9882947020113715,
   'MAPE': 0.6569279076928647},
  'Low': {'MAE': 8.767930520062006,
   'RMSE': 11.643959153532842,
   'R2': 0.9885988122269507,
   'MAPE': 0.6431422575141161}},
 'DF3': {'High': {'MAE': 11.685758561736646,
   'RMSE': 14.955950380155349,
   'R2': 0.75598170285031,
   'MAPE': 0.6494723145273689},
  'Low': {'MAE': 10.229243604548,
   'RMSE': 14.903644356165115,
   'R2': 0.7894882253237724,
   'MAPE': 0.5748607534048208}},
 'DF4': {'High': {'MAE': 344.6612900182147,
   'RMSE': 455.20881097884757,
   'R2': -1.3396454342624273,
   'MAPE': 9.13417843948939},
  'Low': {'MAE': 375.885431959526,
   'RMSE': 471.1035820444804,
   'R2':